In [1]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kmean_path='/mnt/cold1/snaketree/prj/scRNA/dataset/KMeans_2/kmeans/'
panetti_dic={'CRC0322':'CRC0322_NT_1_3000/CRC0322_NT_1_3000_kmeans_2comp.csv',
        'CRC0327':'CRC0327_NT_2/CRC0327_NT_2_kmeans_2comp.csv',
        'CRC0542':'CRC0542_NT72h_1/CRC0542_NT72h_1_kmeans_2comp.csv'}
paneth=pd.DataFrame()
for sample_name in panetti_dic.keys():
    path=os.path.join(kmean_path,panetti_dic[sample_name])
    p=pd.read_csv(path,header=0,index_col=0)
    p.index=p.index+'_'+sample_name
    p=p[['isPaneth']]
    p[p['isPaneth']=='filtered']='nPaneth'
    paneth=pd.concat([paneth,p])

In [4]:
csv_path='NT_GNN_vanilla/'
grafo_path='Graphs/grafo_filtrato_undirect_small.csv'

grafo=pd.read_csv(grafo_path)
geni=grafo['source'].unique()

In [5]:
train_id=['filtered_CRC0327_NT_2.csv','filtered_CRC0542_NT72h_1.csv','filtered_CRC0322_NT_1_3000.csv']
test_id=['filtered_CRC0322_NT_1_3000.csv','filtered_CRC1502_NT_1.csv']
kras=['CRC1502','CRC1620','CRC1139']
wt=['CRC0322','CRC0327','CRC0542']

train=pd.DataFrame()
for file in os.listdir(csv_path):
    sample_name=str.split(file,sep='_')[1]
    if sample_name in kras:
        pass
    else:
        print(sample_name)
        data=pd.read_csv(os.path.join(csv_path,file),header=0,index_col=0)
        valid_geni=[g for g in geni if g in data.columns]
        data=data.loc[:,valid_geni]
        print(len(data.columns))
        data['sample']=sample_name
        data['cell_id']=data.index
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])


CRC0322
939
CRC0327
939
CRC0542
941


In [ ]:
geni_senza_na = train.columns[train.isnull().sum() == 0]
geni_con_na=train.columns[train.isnull().sum() > 0]
print(f"Geni senza NaN: {len(geni_senza_na)}")
print(f"Geni con NaN: {len(geni_con_na)}")
geni = [g for g in geni if g in train.columns]
geni_validi = [
    g for g in geni
    if g in train.columns and
    train[g].isnull().sum() == 0
]

#filtro geni con na
meta_col = [c for c in ['label', 'sample', 'cell_id'] if c in train.columns]
train = train[geni_validi + meta_col]

In [ ]:
train['idx']=train["cell_id"] + "_" + train["sample"]
train.index=train.idx
train.drop(columns='idx',inplace=True)
train=train.loc[paneth.index]

In [ ]:
# expression matrix ==> in anndata X
gene_expr = train.drop(columns=["cell_id", "sample"])
adata = ad.AnnData(X=gene_expr.values)
#meta
adata.obs["cell_id"] = train["cell_id"].values
adata.obs["sample"] = train["sample"].values
adata.obs.index = train.index  # id_univoco per la cellula

adata.var_names = gene_expr.columns

In [ ]:
pca = PCA(n_components=None)
pca.fit(gene_expr)

cumulative_var = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = np.argmax(cumulative_var >= 0.9) + 1
print(f"Componenti per 90% varianza: {n_components_90}")
#varianza cumulativa
cumulative_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% varianza')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% varianza')
plt.xlabel('Numero componente')
plt.ylabel('Varianza cumulativa')
plt.title('Varianza cumulativa (PCA)')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import scanpy as sc
import bbknn
sc.pp.pca(adata,random_state=42,n_comps=30,use_highly_variable=False)
bbknn.bbknn(adata, batch_key='sample')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['sample'])

In [ ]:
res = [0.1, 0.3, 0.4,0.5,0.6]

for r in res:
    leiden_key = f"leiden_{r}"
    louvain_key = f"louvain_{r}"

    sc.tl.leiden(adata, resolution=r, key_added=leiden_key)
    sc.tl.louvain(adata, resolution=r, key_added=louvain_key)


    filename = f"_clustering_r{r}.pdf"  # salverà in ./figures/
    sc.pl.umap(
        adata,
        color=[leiden_key, louvain_key, "sample"],
        save=filename,
        show=True  
    )



In [ ]:
#Marker per Leiden
for r in res:
    key = f'leiden_{r}'
    rank_key = f'rank_{key}'
    sc.tl.rank_genes_groups(adata, groupby=key, method='wilcoxon', key_added=rank_key)
    sc.pl.rank_genes_groups(adata, key=rank_key, n_genes=10, sharey=False, save=f"_{key}.pdf")

In [ ]:
for r in res:
    key = f'louvain_{r}'
    rank_key = f'rank_{key}'
    sc.tl.rank_genes_groups(adata, groupby=key, method='wilcoxon', key_added=rank_key)
    sc.pl.rank_genes_groups(adata, key=rank_key, n_genes=10, sharey=False, save=f"_{key}.pdf")

In [ ]:
adata.obs['kmeans'] = paneth



In [ ]:
clust=pd.DataFrame(adata.obs['leiden_0.6'])

In [ ]:
totale=pd.concat([clust,paneth],axis=1)
totale.columns=['clust','p']

In [ ]:
totale.groupby(['clust', 'p']).size().unstack()

proviamo harmonca

In [ ]:
pca_data = adata.obsm['X_pca']

ho = hm.run_harmony(pca_data, adata.obs, 'sample')  # 'sample' è il batch key

# Salva i nuovi embeddings integrati
adata.obsm['X_pca_harmony'] = ho.Z_corr.T 

In [ ]:
sc.pl.pca(adata, color='sample')  # pre-integrazione
sc.pl.embedding(adata, basis='X_pca_harmony', color='sample') 

In [ ]:
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
sc.tl.leiden(adata, resolution=0.25, key_added='harmony_0.2')
sc.pl.umap(adata, color=['harmony_0.2','sample','kmeans'])
sc.pl.umap(adata, color=['ATOH1','HEPACAM2'])

In [ ]:
clust=pd.DataFrame(adata.obs['harmony_0.2'])
totale=pd.concat([clust,paneth],axis=1)
totale.columns=['clust','p']
totale.groupby(['clust', 'p']).size().unstack()

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='harmony_0.2', method='wilcoxon', key_added='hamony_0.2')
sc.pl.rank_genes_groups(adata, key='hamony_0.2', n_genes=10, sharey=False)

In [ ]:
clust

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='harmony_1.4', method='wilcoxon', key_added='merda')
sc.pl.rank_genes_groups(adata, key='merda', n_genes=10, sharey=False)

In [ ]:
clust=pd.DataFrame(adata.obs['leiden_0.8'])
totale=pd.concat([clust,paneth],axis=1)
totale.columns=['clust','p']
totale.groupby(['clust', 'p']).size().unstack()

In [ ]:
ensg_trad='/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_GSE144735/first_ensg_gs_map.tsv'
ens=pd.read_csv(ensg_trad,sep='\t')
mapping_dict = dict(zip(ens['ens'], ens['gs']))
#leggo dato intero e faccio check su geni variabili
samples=['CRC0322_NT_1_3000','CRC0327_NT_2','CRC0542_NT72h_1']
DIC_FOLDER={
 'CRC0322_NT_1_3000':'rCASC_Ire_cetuxi',
 'CRC0327_NT_2':'rCASC_Ire_cetuxi',
 'CRC0542_NT72h_1':'rCASC_longer'
 }
PRJ_ROOT='/mnt/cold1/snaketree/prj/scRNA'
dic_path={}
data=pd.DataFrame()
i=0
for sample in samples:
    dic_path[sample]=PRJ_ROOT+'/dataset/'+DIC_FOLDER[sample]+'/'+sample+'_dir/annotated_'+sample+'_log2_pc1_cpm.csv'

for sample_name, file_path in dic_path.items():
    print(f"Processing {sample_name}...")
    df = pd.read_csv(file_path,header=0,index_col=0,sep=',').T
    df = df[[col for col in df.columns if col.split(":")[0] in mapping_dict]]
    df.columns = [mapping_dict[col.split(":")[0]] for col in df.columns]
    sample_name=sample_name.split('_')[0]
    df['sample']=sample_name
    df['cell_id']=df.index
    df.reset_index(drop=True,inplace=True)
    print(df.head())
    if i==0:
        data=pd.concat([data,df])
        i=i+1
    else:
        data=pd.concat([data,df],axis=0,join='inner')

print("Done!")

In [ ]:
geni_senza_na = data.columns[data.isnull().sum() == 0]
geni_con_na=data.columns[data.isnull().sum() > 0]
print(f"Geni senza NaN: {len(geni_senza_na)}")
print(f"Geni con NaN: {len(geni_con_na)}")


data['idx']=data["cell_id"] + "_" + data["sample"]
data.index=data.idx
data.drop(columns='idx',inplace=True)
data=data.loc[paneth.index]

In [ ]:
# expression matrix ==> in anndata X
gene_expr = data.drop(columns=["cell_id", "sample"])
adata = ad.AnnData(X=gene_expr.values)
#meta
adata.obs["cell_id"] = data["cell_id"].values
adata.obs["sample"] = data["sample"].values
adata.obs.index = data.index  # id_univoco per la cellula

adata.var_names = gene_expr.columns

In [ ]:
pca = PCA(n_components=None)
pca.fit(gene_expr)

cumulative_var = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = np.argmax(cumulative_var >= 0.9) + 1
print(f"Componenti per 90% varianza: {n_components_90}")

In [ ]:
explained_var = pca.explained_variance_ratio_
cumulative_var = explained_var.cumsum()
#varianza spiegata asse y max 1
plt.figure(figsize=(8,5))
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o')
plt.title('Varianza spiegata da ciascuna componente PCA')
plt.xlabel('Numero componente')
plt.ylabel('Varianza spiegata')
plt.grid(True)
plt.show()

#varianza cumulativa
cumulative_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% varianza')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% varianza')
plt.xlabel('Numero componente')
plt.ylabel('Varianza cumulativa')
plt.title('Varianza cumulativa (PCA)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
sc.pp.pca(adata,random_state=42,n_comps=2000,use_highly_variable=False)
bbknn.bbknn(adata, batch_key='sample')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['sample'])
pca_data = adata.obsm['X_pca']

ho = hm.run_harmony(pca_data, adata.obs, 'sample')  # 'sample' è il batch key

# Salva i nuovi embeddings integrati
adata.obsm['X_pca_harmony'] = ho.Z_corr.T 

In [ ]:
adata.obs['kmeans'] = paneth
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
sc.tl.leiden(adata, resolution=0.6, key_added='merda')
sc.pl.umap(adata, color=['merda','sample','kmeans'])
sc.pl.umap(adata, color=['ATOH1','HEPACAM2'])